# OraShift — QLoRA fine-tuning on a free Kaggle T4

Fine-tunes **Qwen2.5-Coder-3B-Instruct** to translate Oracle SQL into PostgreSQL, using
QLoRA so the whole thing fits in a 16 GB T4.

**Before you run anything**, in the Kaggle sidebar:

| Setting | Value |
| --- | --- |
| Accelerator | **GPU T4 x2** (only one is used, but this is the free tier that has a T4) |
| Internet | **On** (needed to download the model and clone the dataset) |
| Persistence | Variables and files, optional but convenient |

Expected runtime: roughly 25–45 minutes end to end, most of it the model download and
training.

> **This notebook has not been executed by its author.** It was written on a machine with
> no GPU and a network that blocks huggingface.co, so every cell is unrun. It is built to
> fail loudly and early rather than silently: each stage asserts its assumptions and prints
> what it found. If a cell errors, the markdown above it says what the likely cause is.

## 1. Install

Versions are deliberately loose rather than pinned exactly. Unsloth has to match the
CUDA and PyTorch build that Kaggle ships, and that base image changes; a hard pin that
worked last month is the most common way these notebooks break. The next cell records the
exact versions that actually got installed, which is the reproducibility record.

In [ ]:
# Output is deliberately NOT captured, and the install is timed.
#
# A previous run spent 3.6 hours here in silence. The cause was pinning
# "xformers<0.0.30", for which no wheel exists against Kaggle's torch build, so
# pip fell back to compiling it from source. Unsloth resolves its own
# dependencies, so the pin is gone and nothing is hidden.
import time as _t

_start = _t.time()
!pip install -q unsloth
print(f"\ninstall took {(_t.time() - _start) / 60:.1f} minutes")

In [ ]:
import json, os, platform, random, subprocess, time
from pathlib import Path

import torch

SEED = 20260101  # the same seed the dataset generator uses
random.seed(SEED)
torch.manual_seed(SEED)

print("python     ", platform.python_version())
print("torch      ", torch.__version__)
print("cuda avail ", torch.cuda.is_available())

assert torch.cuda.is_available(), "No GPU. In the Kaggle sidebar set Accelerator to 'GPU T4 x2'."

gpu = torch.cuda.get_device_properties(0)
total_gb = gpu.total_memory / 1024**3
print(f"gpu         {gpu.name}  {total_gb:.1f} GB  capability sm_{gpu.major}{gpu.minor}")

# bf16 needs an Ampere or newer GPU, meaning compute capability 8.0+.
#
# Do NOT use torch.cuda.is_bf16_supported() here. It returns True on a T4,
# because PyTorch can *emulate* bf16 on Turing, but transformers checks the
# compute capability and refuses with "You need Ampere+ GPU". A previous run
# died on exactly that, after the model had already loaded.
SUPPORTS_BF16 = gpu.major >= 8
USE_FP16 = not SUPPORTS_BF16
print(f"bf16 (needs sm_80+): {SUPPORTS_BF16}  ->  training in {'fp16' if USE_FP16 else 'bf16'}")

if total_gb < 14:
    print("WARNING: less than 14 GB of VRAM. Reduce MAX_SEQ_LENGTH or batch size below.")

In [ ]:
# The reproducibility record: whatever actually got installed.
for package in [
    "unsloth",
    "torch",
    "transformers",
    "trl",
    "peft",
    "bitsandbytes",
    "accelerate",
    "datasets",
]:
    try:
        module = __import__(package)
        print(f"{package:14} {getattr(module, '__version__', 'unknown')}")
    except ImportError:
        print(f"{package:14} NOT INSTALLED")

## 2. Get the dataset

Cloned straight from the public repository, so the training data always matches what is
committed and anyone reproducing this gets the identical files. The six JSONL splits were
built by `orashift build-dataset`, and every pair in them was verified by **executing**
both statements and comparing results — none of this is text-similarity data.

In [ ]:
REPO_URL = "https://github.com/devansh173/orashift"
REPO_DIR = Path("/kaggle/working/orashift")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

DATA_DIR = REPO_DIR / "data" / "dataset"
assert DATA_DIR.exists(), f"{DATA_DIR} missing. Did the clone succeed?"


def load_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


SPLITS = {
    name: load_jsonl(DATA_DIR / f"{name}.jsonl")
    for name in [
        "train",
        "val",
        "test_in_schema",
        "test_unseen_schema",
        "test_unseen_template",
        "test_unseen_both",
    ]
}

for name, rows in SPLITS.items():
    print(f"{name:22} {len(rows):4d}")

assert SPLITS["train"], "training split is empty"

## 3. Measure the real token lengths

The dataset was built on a machine that could not download this tokenizer, so its token
counts are **estimates** at 3.0 characters per token — deliberately pessimistic. This cell
replaces the estimate with a measurement and reports what actually exceeds the budget.

If the measured maximum is far below 2048, `MAX_SEQ_LENGTH` could be lowered, which would
cut memory use and let the batch size rise.

In [ ]:
# Unsloth must be imported before transformers so it can patch the kernels.
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"

# A previous run measured the real token lengths with this tokenizer: the
# longest example in any split was 804 tokens, against the 2048 the dataset was
# budgeted for. 1024 therefore fits every example with room to spare, and costs
# much less memory and time per step than 2048 would.
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # None lets Unsloth pick fp16 on a T4
    load_in_4bit=True,
)
print("loaded", MODEL_NAME)

In [ ]:
def rendered(example):
    """The example as the model will actually see it, chat template included."""
    return tokenizer.apply_chat_template(example["messages"], tokenize=False)


token_stats = {}
for name, rows in SPLITS.items():
    lengths = [len(tokenizer(rendered(r)).input_ids) for r in rows]
    over = [n for n in lengths if n > MAX_SEQ_LENGTH]
    token_stats[name] = {
        "count": len(lengths),
        "max": max(lengths) if lengths else 0,
        "mean": round(sum(lengths) / len(lengths), 1) if lengths else 0,
        "over_budget": len(over),
    }
    print(
        f"{name:22} n={len(lengths):4d}  max={token_stats[name]['max']:5d}  "
        f"mean={token_stats[name]['mean']:7.1f}  over {MAX_SEQ_LENGTH}: {len(over)}"
    )

estimated_max = max(r["meta"]["estimated_tokens"] for rows in SPLITS.values() for r in rows)
measured_max = max(s["max"] for s in token_stats.values())
print(f"\nestimate said max ~{estimated_max}, measured max is {measured_max}")
verdict = "conservative (good)" if estimated_max >= measured_max else "TOO LOW"
print("the estimate was", verdict)

## 4. Attach the LoRA adapter

**What QLoRA is doing.** The base model's 3.1 billion weights are frozen and stored in
4-bit. Training only touches small adapter matrices inserted beside the big weight
matrices. That is why this fits: optimiser state is proportional to *trainable*
parameters, and Adam keeps two extra copies of each. Around 30 M trainable needs roughly
360 MB of optimiser state; 3.1 B would need about 37 GB, which no T4 will ever hold.

| Setting | Value | What it does, and why this value |
| --- | --- | --- |
| `r` | 16 | Rank of the update. A `d_in x d_out` weight update is approximated by two thin matrices of width `r`, so cost scales with `r`. 8 is often enough for a narrow task, 32+ for a bigger domain shift. 16 is the usual middle for teaching a format and a set of rewrites. |
| `lora_alpha` | 16 | The adapter's output is scaled by `alpha / r`. With `alpha = r` that is 1.0, so the adapter starts at unit scale. Setting `alpha = 2r` effectively doubles the learning rate on the adapter only — a common trick, skipped here to keep one knob doing one job. |
| `lora_dropout` | 0 | Unsloth's fast path requires 0. With 418 examples, overfitting is controlled by the epoch count instead. |
| `target_modules` | all 7 | Attention projections (`q,k,v,o`) plus MLP (`gate,up,down`). Attention-only is cheaper and common, but the MLP is where much of the token-level rewriting lives, and `NVL -> COALESCE` is exactly that kind of rewrite. |
| `use_gradient_checkpointing` | `"unsloth"` | Recomputes activations in the backward pass instead of storing them. Costs ~20% time, saves a lot of memory, and is what makes 2048 tokens possible on a T4. |
| `random_state` | `SEED` | Same seed as the dataset, so the whole pipeline is reproducible. |

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",  # attention
        "gate_proj",
        "up_proj",
        "down_proj",  # MLP
    ],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"trainable : {trainable:,}")
print(f"total     : {total:,}")
print(f"trainable fraction: {trainable / total:.2%}")

## 5. Format the examples

Each example is rendered through Qwen's chat template into a single `text` field. Doing
the rendering here rather than relying on the trainer to do it keeps this working across
TRL versions, which have moved this responsibility around more than once.

In [ ]:
from datasets import Dataset


def to_text_dataset(rows):
    return Dataset.from_list([{"text": rendered(r)} for r in rows])


train_ds = to_text_dataset(SPLITS["train"])
val_ds = to_text_dataset(SPLITS["val"])
print(f"train {len(train_ds)}   val {len(val_ds)}")
print("\n--- one rendered example ---\n")
print(train_ds[0]["text"][:1200])

## 6. Training configuration

| Setting | Value | What it does, and why this value |
| --- | --- | --- |
| `per_device_train_batch_size` | 2 | What fits at 2048 tokens in 4-bit on a 16 GB T4. Raise it only if the measured max length above was well under 2048. |
| `gradient_accumulation_steps` | 4 | Gradients from 4 micro-batches are summed before the optimiser steps, giving an **effective batch of 8** without the memory of a batch of 8. |
| `num_train_epochs` | 3 | 418 examples at effective batch 8 is ~52 steps per epoch, so ~157 steps total. One epoch is not enough to learn the output format; much beyond three starts memorising on a set this small. |
| `learning_rate` | 2e-4 | Roughly 10x what a full fine-tune would use. That is normal for LoRA: only the small adapter moves, so a larger step is safe and needed to make progress in few steps. |
| `lr_scheduler_type` | linear | Decays to zero over training. With so few steps, cosine and linear are near-identical; linear is easier to reason about. |
| `warmup_ratio` | 0.05 | ~8 steps of ramp-up. Prevents a large, badly-scaled first step from disturbing the adapter before Adam's running averages have settled. |
| `optim` | `adamw_8bit` | bitsandbytes 8-bit Adam. Halves optimiser state memory for no measurable quality cost at this scale. |
| `weight_decay` | 0.01 | Mild regularisation. Matters more here than usual because the dataset is small. |
| `fp16` / `bf16` | auto | T4 is Turing and has no bf16, so fp16. Set from the capability check in section 1 rather than hard-coded. |
| `max_seq_length` | 2048 | The dataset's budget. Measured lengths are well inside it. |

In [ ]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/kaggle/working/orashift-qlora"
USE_WANDB = False  # set True and add a WANDB_API_KEY Kaggle secret for a shareable dashboard

if USE_WANDB:
    from kaggle_secrets import UserSecretsClient

    try:
        os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
        os.environ["WANDB_PROJECT"] = "orashift"
        report_to = "wandb"
    except Exception as exc:
        print(f"W&B secret unavailable ({exc}); falling back to tensorboard")
        report_to = "tensorboard"
else:
    report_to = "tensorboard"

print("logging to:", report_to)

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=USE_FP16,
    bf16=not USE_FP16,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    report_to=report_to,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,  # one example per sequence; packing would blur the loss mask below
)

### Loss on assistant tokens only

This is the single most important line in the notebook.

By default the trainer computes loss over **every** token, which means the model is also
being trained to produce the system prompt and the Oracle input — text it will never need
to generate. That wastes capacity and measurably degrades the output.

`train_on_responses_only` masks everything before `<|im_start|>assistant` out of the loss,
so gradients come only from the PostgreSQL translation. Those markers are Qwen's ChatML
template; a different base model would need different ones.

The cell after it decodes one masked example and prints only the tokens that still carry
loss. If that output is not just a PostgreSQL statement, the masking is wrong and training
would quietly learn the wrong thing.

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=args,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print("loss restricted to assistant tokens")

In [ ]:
# Verify the mask. Only the translation should survive.
sample = trainer.train_dataset[0]
labels = sample["labels"]
kept = [t for t, label in zip(sample["input_ids"], labels, strict=True) if label != -100]
print("tokens carrying loss:\n")
print(tokenizer.decode(kept))
print("\n(if the above shows the system prompt or the Oracle input, the mask is wrong)")

## 7. Train

In [ ]:
start = time.time()
train_result = trainer.train()
minutes = (time.time() - start) / 60

print(f"\ntrained in {minutes:.1f} minutes")
print("final train loss:", round(train_result.training_loss, 4))

# Save straight away, before anything else can fail.
#
# A previous run trained successfully for 12.9 minutes and then died on the
# next line, losing the adapter entirely because saving came later in the
# notebook. The expensive, irreversible work is finished at this point, so it
# gets persisted at this point.
ADAPTER_DIR = "/kaggle/working/orashift-qlora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"adapter saved to {ADAPTER_DIR}")

# Do NOT call trainer.evaluate() here. eval_strategy="epoch" has already run
# evaluation during training and the results are in the log history. Calling it
# again after train() has finished raises
# "on_train_begin must be called before on_evaluate", because the logging
# callback's tracker has already been torn down.
eval_losses = [entry["eval_loss"] for entry in trainer.state.log_history if "eval_loss" in entry]
final_eval_loss = eval_losses[-1] if eval_losses else None
print("eval loss per epoch:", [round(v, 4) for v in eval_losses])
print("final eval loss    :", round(final_eval_loss, 4) if final_eval_loss else "n/a")

## 8. Sanity check

Not a score — scoring happens in phase 6 on the full test sets, by executing the SQL.
This is only asking "has it learned anything at all" before spending another notebook on
inference.

Two examples from each of the four test buckets, greedy decoding so the output is
deterministic. Read the predictions against the expected column: at this stage the useful
signal is whether the output is a bare PostgreSQL statement with the Oracle-isms gone, not
whether it is character-perfect.

In [ ]:
FastLanguageModel.for_inference(model)


def translate(example, max_new_tokens=256):
    prompt_messages = example["messages"][:2]  # system + user, not the answer
    prompt = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy: reproducible
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1] :]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


for split in ["test_in_schema", "test_unseen_schema", "test_unseen_template", "test_unseen_both"]:
    print("=" * 100)
    print(split.upper())
    print("=" * 100)
    for example in SPLITS[split][:2]:
        print(f"\ncategory : {example['meta']['category']}")
        print(f"oracle   : {example['messages'][1]['content'][:180]}")
        print(f"expected : {example['messages'][2]['content'][:180]}")
        print(f"predicted: {translate(example)[:180]}")
    print()

## 9. Save the adapter and the run record

Only the adapter is saved — about 60 MB against the base model's several gigabytes. That
is the other advantage of LoRA: what you ship is the diff, not a whole model.

`train_run.json` holds the measured facts from this run. Download it into `results/` in
the repository so phase 7 can cite real numbers rather than remembered ones.

In [ ]:
# Already saved immediately after training; this only reports on it.
size_mb = sum(f.stat().st_size for f in Path(ADAPTER_DIR).rglob("*") if f.is_file()) / 1024**2
print(f"adapter saved to {ADAPTER_DIR}  ({size_mb:.1f} MB)")
print(sorted(p.name for p in Path(ADAPTER_DIR).iterdir()))

In [ ]:
run_record = {
    "base_model": MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "seed": SEED,
    "precision": "fp16" if USE_FP16 else "bf16",
    "gpu": torch.cuda.get_device_properties(0).name,
    "lora": {
        "r": 16,
        "alpha": 16,
        "dropout": 0,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    },
    "trainable_parameters": int(trainable),
    "total_parameters": int(total),
    "trainable_fraction": round(trainable / total, 6),
    "hyperparameters": {
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "effective_batch_size": 8,
        "num_train_epochs": 3,
        "learning_rate": 2e-4,
        "lr_scheduler_type": "linear",
        "warmup_ratio": 0.05,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
    },
    "split_sizes": {name: len(rows) for name, rows in SPLITS.items()},
    "token_stats": token_stats,
    "final_train_loss": round(float(train_result.training_loss), 4),
    "eval_loss_per_epoch": [round(float(v), 4) for v in eval_losses],
    "final_eval_loss": round(float(final_eval_loss), 4) if final_eval_loss else None,
    "train_minutes": round(minutes, 1),
    "adapter_mb": round(size_mb, 1),
}

with open("/kaggle/working/train_run.json", "w", encoding="utf-8") as handle:
    json.dump(run_record, handle, indent=2)

print(json.dumps(run_record, indent=2))

## 10. Push the adapter to the Hub (optional)

Set `HF_USERNAME` and add an `HF_TOKEN` Kaggle secret with **write** access. Skipped
entirely if the secret is missing, so it cannot fail a run that otherwise succeeded.

In [ ]:
HF_USERNAME = ""  # e.g. "devansh173"
HF_REPO = f"{HF_USERNAME}/orashift-qwen2.5-coder-3b-qlora" if HF_USERNAME else None

if HF_REPO:
    try:
        from kaggle_secrets import UserSecretsClient

        hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        model.push_to_hub(HF_REPO, token=hf_token)
        tokenizer.push_to_hub(HF_REPO, token=hf_token)
        print(f"pushed to https://huggingface.co/{HF_REPO}")
    except Exception as exc:
        print(f"push skipped: {exc}")
else:
    print("HF_USERNAME not set; skipping the Hub push")

## What to download before this session ends

Kaggle deletes `/kaggle/working` when the session is reclaimed. From the output panel, save:

1. **`orashift-qlora-adapter/`** — the trained adapter. Phase 6 needs it.
2. **`train_run.json`** — put it in `results/` in the repository.

Then say so, and phase 6 (inference over the full test sets) can start.